# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will:
- Load Croissant metadata from a URL,
- Review record sets and field `@id`s,
- Load tabular data by referencing `@id`s only,
- Perform exploratory data analysis (EDA),
- Visualize select attributes.

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- See dataset at [SEN Science](https://sen.science/doi/10.71728/senscience.qs2f-h81p)

> **Note:** All dataset entities (record sets, fields, columns, etc.) are referenced by their `@id` only, as per best practice and Croissant specification.

In [ ]:
# Install the mlcroissant library if not already present
!pip install mlcroissant

## 1. Data Loading

Load dataset Croissant metadata and records. The Croissant package provides seamless access to metadata and data defined by the schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Initialize the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic information
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Number of record sets: {len(metadata.record_sets)}")

## 2. Data Overview

List all available record sets in the dataset, with their `@id` (unique identifier), label/name, and a brief description. Each record set corresponds to a table or collection of related records.

For each record set, list the available fields and their `@id`, type, and description. This helps decide what data to analyze.

In [ ]:
# List all record sets and their details by @id
for rset in metadata.record_sets:
    print(f"Record Set: {rset['@id']}")
    print(f"  name: {rset.get('name','N/A')}")
    print(f"  description: {rset.get('description','')}")
    print("  Fields:")
    for field in rset.get('fields', []):
        print(f"    - {field['@id']} (type: {field.get('dataType', 'N/A')}, name: {field.get('name', '')}) -- {field.get('description','')}")
    print("\n------------------------\n")

## 3. Data Extraction

Extract data for each record set using its `@id` as defined above. Data are loaded into pandas DataFrames, keyed by the record set `@id`. List the columns for reference.

> **Note:** Replace the `<RECORD_SET_ID>` below with the exact string of the primary record set's `@id`.

In [ ]:
# List of record set @ids
record_set_ids = [rset['@id'] for rset in metadata.record_sets]

dataframes = {}
# Load all available record sets into memory
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

for rid in dataframes:
    print(f"Columns in {rid}: {dataframes[rid].columns.tolist()}")

# Choose a main record set (use the first, or set manually if known)
main_record_set_id = record_set_ids[0] if record_set_ids else None

# Display preview of main record set
if main_record_set_id:
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Select a numeric field and a group (categorical) field by their `@id` (as previously listed). Filter, normalize, and aggregate as standard EDA steps. You can adjust the threshold and field names to fit the data.

In [ ]:
# Example: Select numeric and group field @ids from the main record set
# Inspect column names to identify a numeric field such as 'Age', and a group field such as 'Sex' or 'MSI_status'
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Available columns: {df.columns.tolist()}")
    # Example field IDs (replace with actual @id as found in earlier step)
    # Numeric field: use the @id or column name string
    numeric_field_id = None
    group_field_id = None

    # Attempt to infer likely numeric/group fields by column name
    for col in df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
        if 'sex' in col.lower() or 'msi' in col.lower() or 'group' in col.lower():
            group_field_id = col

    if numeric_field_id:
        # Remove outliers below a threshold and normalize
        threshold = 40  # e.g., filter for age > 40
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df)
    else:
        print("No numeric field identified for EDA.")

## 5. Visualization

Visualize the numeric field's distribution and any potential group-wise differences. Here, we use matplotlib for quick plots. Adjust fields and labels according to your variable selection above.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,5))
    df[numeric_field_id].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, you used the `mlcroissant` library to load, inspect, and analyze the FAIR² colorectal cancer survivors dataset by referencing all record sets and fields by their `@id`. You previewed the schema, explored data, filtered and normalized a key numeric field, and visualized clinical variables. Continue analysis as needed using the column and record set IDs provided.

For more information on the dataset, see the [SEN Science DOI entry](https://sen.science/doi/10.71728/senscience.qs2f-h81p/) and the [mlcroissant documentation](https://mlcommons.github.io/croissant/).